# THUMOS Baseline Report

This notebook is a report notebook for the THUMOS baseline workflow.

Use `modal_pipeline/baseline_workflow.py` for the actual Modal orchestration. This notebook is only for launching that workflow from Jupyter when needed and for reviewing the returned metrics and plots inline.

It can:
- launch the Modal baseline workflow,
- cache the returned run manifest locally for reuse,
- display the key accuracy metrics,
- render training and validation plots inline.


In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt

from modal_pipeline.baseline_workflow import run_baseline_workflow

CONFIG_PATH = "configs/base_config.yaml"
BACKBONE_NAME = ""
SKIP_FEATURE_EXTRACTION = False
RUN_MODAL_WORKFLOW = False
LOCAL_MANIFEST = Path("last_run_manifest.json")


ModuleNotFoundError: No module named 'modal'

In [ ]:
if RUN_MODAL_WORKFLOW:
    run_outputs = run_baseline_workflow(
        config_path=CONFIG_PATH,
        backbone_name=BACKBONE_NAME,
        skip_feature_extraction=SKIP_FEATURE_EXTRACTION,
    )
    LOCAL_MANIFEST.write_text(json.dumps(run_outputs, indent=2), encoding="utf-8")
    print(f"Saved run manifest to {LOCAL_MANIFEST.resolve()}")
else:
    if not LOCAL_MANIFEST.exists():
        raise FileNotFoundError(
            "No cached manifest found. Set RUN_MODAL_WORKFLOW = True to launch a run first."
        )
    run_outputs = json.loads(LOCAL_MANIFEST.read_text(encoding="utf-8"))

list(run_outputs.keys())


In [ ]:
training_summary = run_outputs["training_summary"]
evaluation_summary = run_outputs["evaluation_summary"]
history_summary = run_outputs["history_summary"]

print("Best metric:", training_summary["best_metric_name"])
print("Best value:", training_summary["best_metric_value"])
print("Epochs:", training_summary["epochs"])
print()
print("Evaluation summary")
for key in ["binary_accuracy", "precision", "recall", "f1", "val_loss"]:
    print(f"  {key}: {evaluation_summary.get(key)}")
print()
print("Best epoch snapshot")
print(json.dumps(history_summary.get("best_epoch", {}), indent=2))


In [ ]:
history = run_outputs["history"]
epochs = [row["epoch"] for row in history]
train_loss = [row["train_loss"] for row in history]
val_loss = [row["val_loss"] for row in history]
val_f1 = [row["val_f1"] for row in history]
val_accuracy = [row["val_binary_accuracy"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_loss, label="train_loss")
axes[0].plot(epochs, val_loss, label="val_loss")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs, val_f1, label="val_f1")
axes[1].plot(epochs, val_accuracy, label="val_binary_accuracy")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Metric")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
metric_names = ["binary_accuracy", "precision", "recall", "f1"]
metric_values = [evaluation_summary.get(name, 0.0) for name in metric_names]

plt.figure(figsize=(8, 5))
plt.bar(metric_names, metric_values)
plt.ylim(0.0, 1.0)
plt.ylabel("Score")
plt.title("Evaluation Summary")
plt.show()
